In [115]:
import re, os, json
from unidecode import unidecode
import pandas as pd

In [116]:
gold_str = "Frondibus, et null\u00e2 lucos agitante procell\u00e2,"

answer_str_1 = "Et procellā nullā lucos agitante frondibus."

In [117]:
data_dir = '../data/final_dataset'
long_ans_files = [
                  'prosody_caesura_scansion_english.json',
                  'prosody_caesura_scansion_latin.json',
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

prosody_caesura_scansion_english.json 41
prosody_caesura_scansion_latin.json 41


In [151]:
file_to_resp = {}
#model_name = 'llama3-turbo'
model_name = 'qwq'

model_resp_dir = f'../data/model_responses/{model_name}'

for file in file_to_data.keys():
    base_name = os.path.basename(file)
    file_to_resp[base_name] = {}
    with open(os.path.join(model_resp_dir, base_name), 'r') as f:
        file_to_resp[base_name] = json.load(f)
    print(base_name, len(file_to_resp[base_name]))


prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [119]:
def parse_resp(resp_text):
    '''get the answer from the full response. usually on the last line as Answer: answer'''

    lines = resp_text.split('\n')
    # usually in last line
    line = lines[-1]
    # look for "Answer: A"
    if 'Answer:' in line:
        answer =line.split('Answer:')[1].strip()
        return answer
    else:
        # try to look backwards in the lines
        for line in reversed(lines):
            if 'Answer:' in line:
                answer = line.split('Answer:')[1].strip()
                return answer
        print('error parsing')
        print(resp_text)
        return ''

def normalize_str(s):
    # no accents
    s = unidecode(s)
    
    s = s.lower().strip()
    s = re.sub(r'[^\w\s]', '', s)
    return s

def score(gold_str, answer_str):
    gold_str = normalize_str(gold_str)
    answer_str = normalize_str(answer_str)

    gold_words = gold_str.split()
    answer_words = answer_str.split()

    if len(gold_words) != len(answer_words):
        return 0
    
    print('gold:', gold_words)
    print('pred:', answer_words)
    
    # position of each word must be the same
    for i, word in enumerate(gold_words):
        if word != answer_words[i]:
            return 0
        
    
    
    return 1
    
    

In [120]:
# make dfs
question_dfs = []
for file in file_to_data.keys():
    questions = file_to_data[file]
    question_df = pd.DataFrame(questions)
    question_dfs.append(question_df)

In [121]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    answer_dict = file_to_resp[file]
    answer_col = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']

        model_answer = answer_dict[q_id]
        answer_col.append(model_answer)

    question_df['raw_resp'] = answer_col

In [122]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['parsed_resp'] = question_df['raw_resp'].apply(parse_resp)
    question_dfs[i] = question_df
question_dfs[0]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question_text,multiple_choice_options,answers,raw_resp,parsed_resp
0,exercises-in-latin-prosody,1823,prosody_III.I.9.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Ipse dei clypeus terrâ cùm tollitur imâ,]","<think>\n\nOkay, so I need to adjust the posit...","Ipse dei clypeus imâ terrâ cùm tollitur,"
1,exercises-in-latin-prosody,1823,prosody_III.I.9.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Manè rubet; terrâque rubet cùm conditur imâ.],"<think>\n\nOkay, so I need to adjust the given...","Manè rubet; terrâque rubet, cùm conditur imâ."
2,exercises-in-latin-prosody,1823,prosody_III.I.10.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[En, proles antiqua redit; concordia, virtus,]","<think>\n\nOkay, let's see. The user wants me ...","En, proles antiqua redit virtus, concordia;"
3,exercises-in-latin-prosody,1823,prosody_III.I.10.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Cumque fide pietas altâ cervice vagantur.],"<think>\n\nOkay, so the user wants me to adjus...",Cumque fide pietas altâ cervice vagantur.
4,exercises-in-latin-prosody,1823,prosody_III.I.11.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Robora nec cuneis, et olentem scindere cedrum,]","<think>\n\nOkay, let's see. The user wants me ...","Robora nec cuneis, olentem *et* scindere cedrum,"
5,exercises-in-latin-prosody,1823,prosody_III.I.11.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Nec plaustris cessant vectare gementibus ornos.],"<think>\n\nOkay, let's tackle this problem. Th...",Nec plaustris cessant vectare gementibus *ornos*.
6,exercises-in-latin-prosody,1823,prosody_III.I.12.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Sponte tuus florebit ager cessante juvenco;],"<think>\n\nOkay, so the user wants me to adjus...",Sponte tuus florebit ager cessante juvenco;
7,exercises-in-latin-prosody,1823,prosody_III.I.12.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Ditior oblatas mirabitur incola messes.],"<think>\n\nOkay, let's tackle this problem. Th...",Oblatas mirabitur incola ditior messes.
8,exercises-in-latin-prosody,1823,prosody_III.I.13.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Non propter vitam faciunt patrimonia quidam,]","<think>\n\nOkay, so the user wants me to take ...","Non propter vitam faciunt quidam patrimonia,"
9,exercises-in-latin-prosody,1823,prosody_III.I.13.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Sed, vitio cæci, propter patrimonia vivunt.]","<think>\nOkay, let's tackle this problem. The ...","Vitio cæci, propter patrimonia sed vivunt."


In [123]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    scores = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']
        correct_answer = row['answers'][0]
        model_answer = row['parsed_resp']
        scores.append(score(correct_answer, model_answer)) 

    question_df['score'] = scores
question_dfs[0]

gold: ['ipse', 'dei', 'clypeus', 'terra', 'cum', 'tollitur', 'ima']
pred: ['ipse', 'dei', 'clypeus', 'ima', 'terra', 'cum', 'tollitur']
gold: ['mane', 'rubet', 'terraque', 'rubet', 'cum', 'conditur', 'ima']
pred: ['mane', 'rubet', 'terraque', 'rubet', 'cum', 'conditur', 'ima']
gold: ['en', 'proles', 'antiqua', 'redit', 'concordia', 'virtus']
pred: ['en', 'proles', 'antiqua', 'redit', 'virtus', 'concordia']
gold: ['cumque', 'fide', 'pietas', 'alta', 'cervice', 'vagantur']
pred: ['cumque', 'fide', 'pietas', 'alta', 'cervice', 'vagantur']
gold: ['robora', 'nec', 'cuneis', 'et', 'olentem', 'scindere', 'cedrum']
pred: ['robora', 'nec', 'cuneis', 'olentem', 'et', 'scindere', 'cedrum']
gold: ['nec', 'plaustris', 'cessant', 'vectare', 'gementibus', 'ornos']
pred: ['nec', 'plaustris', 'cessant', 'vectare', 'gementibus', 'ornos']
gold: ['sponte', 'tuus', 'florebit', 'ager', 'cessante', 'juvenco']
pred: ['sponte', 'tuus', 'florebit', 'ager', 'cessante', 'juvenco']
gold: ['ditior', 'oblatas', 'mir

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question_text,multiple_choice_options,answers,raw_resp,parsed_resp,score
0,exercises-in-latin-prosody,1823,prosody_III.I.9.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Ipse dei clypeus terrâ cùm tollitur imâ,]","<think>\n\nOkay, so I need to adjust the posit...","Ipse dei clypeus imâ terrâ cùm tollitur,",0
1,exercises-in-latin-prosody,1823,prosody_III.I.9.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Manè rubet; terrâque rubet cùm conditur imâ.],"<think>\n\nOkay, so I need to adjust the given...","Manè rubet; terrâque rubet, cùm conditur imâ.",1
2,exercises-in-latin-prosody,1823,prosody_III.I.10.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[En, proles antiqua redit; concordia, virtus,]","<think>\n\nOkay, let's see. The user wants me ...","En, proles antiqua redit virtus, concordia;",0
3,exercises-in-latin-prosody,1823,prosody_III.I.10.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Cumque fide pietas altâ cervice vagantur.],"<think>\n\nOkay, so the user wants me to adjus...",Cumque fide pietas altâ cervice vagantur.,1
4,exercises-in-latin-prosody,1823,prosody_III.I.11.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Robora nec cuneis, et olentem scindere cedrum,]","<think>\n\nOkay, let's see. The user wants me ...","Robora nec cuneis, olentem *et* scindere cedrum,",0
5,exercises-in-latin-prosody,1823,prosody_III.I.11.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Nec plaustris cessant vectare gementibus ornos.],"<think>\n\nOkay, let's tackle this problem. Th...",Nec plaustris cessant vectare gementibus *ornos*.,1
6,exercises-in-latin-prosody,1823,prosody_III.I.12.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Sponte tuus florebit ager cessante juvenco;],"<think>\n\nOkay, so the user wants me to adjus...",Sponte tuus florebit ager cessante juvenco;,1
7,exercises-in-latin-prosody,1823,prosody_III.I.12.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Ditior oblatas mirabitur incola messes.],"<think>\n\nOkay, let's tackle this problem. Th...",Oblatas mirabitur incola ditior messes.,0
8,exercises-in-latin-prosody,1823,prosody_III.I.13.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Non propter vitam faciunt patrimonia quidam,]","<think>\n\nOkay, so the user wants me to take ...","Non propter vitam faciunt quidam patrimonia,",0
9,exercises-in-latin-prosody,1823,prosody_III.I.13.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Sed, vitio cæci, propter patrimonia vivunt.]","<think>\nOkay, let's tackle this problem. The ...","Vitio cæci, propter patrimonia sed vivunt.",0


In [124]:
# create combined df, with new columns called "source file"
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['source_file'] = [file] * len(question_df)
    question_dfs[i] = question_df

combined_df = pd.concat(question_dfs)
combined_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question_text,multiple_choice_options,answers,raw_resp,parsed_resp,score,source_file
0,exercises-in-latin-prosody,1823,prosody_III.I.9.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Ipse dei clypeus terrâ cùm tollitur imâ,]","<think>\n\nOkay, so I need to adjust the posit...","Ipse dei clypeus imâ terrâ cùm tollitur,",0,prosody_caesura_scansion_english.json
1,exercises-in-latin-prosody,1823,prosody_III.I.9.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Manè rubet; terrâque rubet cùm conditur imâ.],"<think>\n\nOkay, so I need to adjust the given...","Manè rubet; terrâque rubet, cùm conditur imâ.",1,prosody_caesura_scansion_english.json
2,exercises-in-latin-prosody,1823,prosody_III.I.10.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[En, proles antiqua redit; concordia, virtus,]","<think>\n\nOkay, let's see. The user wants me ...","En, proles antiqua redit virtus, concordia;",0,prosody_caesura_scansion_english.json
3,exercises-in-latin-prosody,1823,prosody_III.I.10.2,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],[Cumque fide pietas altâ cervice vagantur.],"<think>\n\nOkay, so the user wants me to adjus...",Cumque fide pietas altâ cervice vagantur.,1,prosody_caesura_scansion_english.json
4,exercises-in-latin-prosody,1823,prosody_III.I.11.1,short_answer,scansion,unknown,english,latin,Form the following line into hexameter or pent...,[],"[Robora nec cuneis, et olentem scindere cedrum,]","<think>\n\nOkay, let's see. The user wants me ...","Robora nec cuneis, olentem *et* scindere cedrum,",0,prosody_caesura_scansion_english.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36,exercises-in-latin-prosody,1823,prosody_III.I.23.4,short_answer,scansion,unknown,latin,latin,Versum infra positum in hexametrum aut pentame...,[],[Si comportatis rebus bene cogitat uti.],"<think>\nOkay, the user wants me to turn the g...",Si uti compositis rebus bene cogitat.**,0,prosody_caesura_scansion_latin.json
37,exercises-in-latin-prosody,1823,prosody_III.I.24.1,short_answer,scansion,unknown,latin,latin,Versum infra positum in hexametrum aut pentame...,[],"[. Hic sedes augusta deæ, templique colendi]","<think>\n\nOkay, so the user wants me to turn ...","Hic sedes augusta deæ, cultus templique.**",0,prosody_caesura_scansion_latin.json
38,exercises-in-latin-prosody,1823,prosody_III.I.24.2,short_answer,scansion,unknown,latin,latin,Versum infra positum in hexametrum aut pentame...,[],"[Religiosa silex, densis quam pinus obumbrat]","<think>\n\nOkay, let's tackle this problem. Th...","Silex religiosa, quam pinus densis obumbrat.",0,prosody_caesura_scansion_latin.json
39,exercises-in-latin-prosody,1823,prosody_III.I.24.3,short_answer,scansion,unknown,latin,latin,Versum infra positum in hexametrum aut pentame...,[],"[Frondibus, et nullâ lucos agitante procellâ,]","<think>\nOkay, the user wants me to translate ...","Frondibus, et nullâ procellâ lucos agitante,",0,prosody_caesura_scansion_latin.json


In [125]:
normalize_str(answer_str_1)

'et procella nulla lucos agitante frondibus'

In [126]:
score(gold_str, answer_str_1)

gold: ['frondibus', 'et', 'nulla', 'lucos', 'agitante', 'procella']
pred: ['et', 'procella', 'nulla', 'lucos', 'agitante', 'frondibus']


0

In [127]:
# save combined df
save_dir = f'../data/model_responses_parsed/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# drop the columns question, multiple_choice_options, n_required, correctness_logic
save_df = combined_df.drop(columns=['question_text', 'multiple_choice_options', ])
save_dict = save_df.to_dict(orient='records')



with open(os.path.join(save_dir, 'prosody_caesura.json'), 'w') as f:
    json.dump(save_dict, f, indent=4)

In [128]:
def accuracy_by(df, group_cols):
    """
    df          : DataFrame with a single 'score' column (0/1 values)
    group_cols  : column or list of columns to group on
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    return (
        df
        .groupby(group_cols, dropna=False)['score']
        .agg(accuracy='mean', n_items='size')
        .reset_index()
    )

In [129]:
acc_lang_combos = accuracy_by(combined_df,
                             group_cols=["question_language", "answer_language"],
                             #score_cols=model_name+'_score'
                             )
acc_lang_combos

,question_language,answer_language,accuracy,n_items
0,english,latin,0.121951,41
1,latin,latin,0.146341,41


feet questions

In [154]:
with_instr = False

In [152]:
data_dir = '../data/final_dataset'
long_ans_files = [
                  'prosody_feet_questions_english.json',
                  'prosody_feet_questions_latin.json',
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [155]:
file_to_resp = {}

model_resp_dir = f'../data/model_responses/{model_name}'

for file in file_to_data.keys():
    base_name = os.path.basename(file)
    
    if with_instr:
        base_name = base_name.replace('.json', '_with_instr.json')
    file_to_resp[base_name] = {}
    with open(os.path.join(model_resp_dir, base_name), 'r') as f:
        file_to_resp[base_name] = json.load(f)
    print(base_name, len(file_to_resp[base_name]))


prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [156]:
# make dfs
question_dfs = []
for file in file_to_data.keys():
    questions = file_to_data[file]
    question_df = pd.DataFrame(questions)
    question_dfs.append(question_df)

In [157]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    if with_instr:
        file = file.replace('.json', '_with_instr.json')
    answer_dict = file_to_resp[file]
    answer_col = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']

        model_answer = answer_dict[q_id]
        answer_col.append(model_answer)

    question_df['raw_resp'] = answer_col


In [158]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    question_df['parsed_resp'] = question_df['raw_resp'].apply(parse_resp)
    question_dfs[i] = question_df
question_dfs[0]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp
0,exercises-in-latin-prosody,1823,prosody_app.I.1.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so the user is asking about t...","iamb, spondee, iamb, spondee, spondee, trochee**"
1,exercises-in-latin-prosody,1823,prosody_app.I.1.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so I need to figure out the f...","Trochee,Cretic,Dactyl,Trochee,Bacchius"
2,exercises-in-latin-prosody,1823,prosody_app.I.2.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's see. The user provided ...","iamb,iamb,iamb,spondee,spondee,iamb"
3,exercises-in-latin-prosody,1823,prosody_app.I.2.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's tackle this question. T...","Spondee,Trochee,Spondee,Spondee"
4,exercises-in-latin-prosody,1823,prosody_app.I.3.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's tackle this question. The...","pyrrhic, cretic, spondee"
5,exercises-in-latin-prosody,1823,prosody_app.I.3.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user is asking me to i...","spondee,trochee,iamb,iamb"
6,exercises-in-latin-prosody,1823,prosody_app.I.3.3,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's see. The user wants me to...","Trochee,Choriamb,Iamb"
7,exercises-in-latin-prosody,1823,prosody_app.I.3.4,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user provided a line o...","trochee, pyrrhic, trochee, pyrrhic"
8,exercises-in-latin-prosody,1823,prosody_app.I.4.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Choriamb, Amphibrac]","<think>\n\nOkay, let's see. The user provided ...","spondee, spondee, iamb"
9,exercises-in-latin-prosody,1823,prosody_app.I.4.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Second Epitrit, Choriamb, Choriamb, Amphibrac]","<think>\n\nOkay, so I need to figure out the m...","molossus, spondee, iamb, dactyl, dactyl, trochee"


In [159]:
gold_names = set()
pred_names = set()

for question_df in question_dfs:
    answers = question_df['answers'].to_list()
    answer_strs = [a[0] for  a in answers]
    for a in answer_strs:
        a = a.lower()
        names = a.split(',')
        names = [n.strip() for n in names]
        gold_names.update(names)

    parsed_resps = question_df['parsed_resp'].to_list()
    for a in parsed_resps:
        a = a.lower()
        names = a.split(',')
        names = [n.strip() for n in names]
        pred_names.update(names)




In [136]:
gold_names

{'amphibrac',
 'anapest',
 'choriamb',
 'choriambus',
 'dactyl',
 'dichoree',
 'great ionic',
 'iambus',
 'pyrrhic',
 'second epitrit',
 'small ionic',
 'spondee',
 'tribrac',
 'trochee'}

In [102]:
for g in sorted(list(gold_names)):
    print(g, end=', ')

amphibrac, anapest, choriamb, choriambus, dactyl, dichoree, great ionic, iambus, pyrrhic, second epitrit, small ionic, spondee, tribrac, trochee, 

In [160]:
pred_names

{'amphibrach',
 'anapest',
 'bacchius',
 'choriamb',
 'cretic',
 'dactyl',
 'dactylus',
 'iamb',
 'iambus',
 'molossus',
 'pyrrhic',
 'spondee',
 'spondeus',
 'tetrabrach',
 'tribrach',
 'trochee',
 'trochee**',
 'trocheus'}

In [168]:
# mapping for alternate names
mapping = {
    'dactyl' : ['dactylus'],
    'dactylus': ['dactyl'],
    'iamb': ['iambus'],
    'iambus': ['iamb'],
    'spondaeus': ['spondee', 'spondeus'],
    'spondee': ['spondaeus', 'spondeus'],
    'spondeus': ['spondaeus', 'spondee'],
    'trochaeus': ['trochee', 'trocheus'],
    'trochee': ['trochaeus', 'trocheus'],
    'trocheus': ['trochaeus', 'trochee'],

    'amphibrac': ['amphibrach'],
    'amphibrach': ['amphibrac'],
    'anapest': [],
    'bacchius': [],
    'choriamb': ['choriambus'],
    'choriambus': ['choriamb'],
    'cretic': [],
    'dichoree': [],
    'great ionic': [],
    'molossus': [],
    'pyrrhic': [],
    'second epitrit': [],
    'small ionic': [],
    'tribrac': ['tribrach'],
    'tribrach': ['tribrac'],
    'tetrabrach': [],
}

In [166]:
def score_feet(gold_str, answer_str):
    gold_str = gold_str.lower()
    answer_str = answer_str.lower()

    answer_str = answer_str.replace('*', '')

    gold_words = gold_str.split(',')
    answer_words = answer_str.split(',')
    gold_words = [w.strip() for w in gold_words]
    answer_words = [w.strip() for w in answer_words]

    #if len(gold_words) != len(answer_words):
    #    return 0
    
    print('gold:', gold_words)
    print('pred:', answer_words)
    
    # position of each word must be the same
    correct = 0
    total = 0
    for (word, pred) in zip(gold_words, answer_words):
        if word == pred: correct +=1
        elif any(word==p for p in mapping[pred]): correct += 1

        total += 1
    
    if total != len(gold_words): total = len(gold_words)
    
    return correct / total if total > 0 else 0

In [106]:
question_dfs[0].iloc[0]

source_name                                       exercises-in-latin-prosody
source_year                                                             1823
question_id                                                prosody_app.I.1.1
question_format                                                 short_answer
question_content                                                    scansion
difficulty                                                           unknown
question_language                                                    english
answer_language                                                      english
question                   What is the name of each foot in the following...
multiple_choice_options                                                   []
answers                         [Trochee, Spondee, Dactyl, Trochee, Trochee]
raw_resp                   To identify the feet in the given line of poet...
parsed_resp                        trochee, trochee, dactyl, dactyl, spondee

In [163]:
gold = question_dfs[0].iloc[0]['answers'][0]
pred = question_dfs[0].iloc[0]['parsed_resp']
score_feet(gold, pred)

gold: ['trochee', 'spondee', 'dactyl', 'trochee', 'trochee']
pred: ['iamb', 'spondee', 'iamb', 'spondee', 'spondee', 'trochee']


0.2

In [169]:
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    scores = []
    for (i, row) in question_df.iterrows():
        q_id = row['question_id']
        correct_answer = row['answers'][0]
        model_answer = row['parsed_resp']
        answer_language = row['answer_language']
        scores.append(score_feet(correct_answer, model_answer))

    question_df['score'] = scores
question_dfs[0]

gold: ['trochee', 'spondee', 'dactyl', 'trochee', 'trochee']
pred: ['iamb', 'spondee', 'iamb', 'spondee', 'spondee', 'trochee']
gold: ['trochee', 'spondee', 'dactyl', 'trochee', 'trochee']
pred: ['trochee', 'cretic', 'dactyl', 'trochee', 'bacchius']
gold: ['iambus', 'iambus', 'iambus', 'iambus', 'spondee', 'iambus']
pred: ['iamb', 'iamb', 'iamb', 'spondee', 'spondee', 'iamb']
gold: ['iambus', 'iambus', 'spondee', 'iambus']
pred: ['spondee', 'trochee', 'spondee', 'spondee']
gold: ['spondee', 'choriambus', 'iambus']
pred: ['pyrrhic', 'cretic', 'spondee']
gold: ['spondee', 'choriambus', 'choriambus', 'iambus']
pred: ['spondee', 'trochee', 'iamb', 'iamb']
gold: ['spondee', 'choriambus', 'iambus']
pred: ['trochee', 'choriamb', 'iamb']
gold: ['spondee', 'choriambus', 'choriambus', 'iambus']
pred: ['trochee', 'pyrrhic', 'trochee', 'pyrrhic']
gold: ['choriamb', 'amphibrac']
pred: ['spondee', 'spondee', 'iamb']
gold: ['second epitrit', 'choriamb', 'choriamb', 'amphibrac']
pred: ['molossus', 'sp

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp,score
0,exercises-in-latin-prosody,1823,prosody_app.I.1.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so the user is asking about t...","iamb, spondee, iamb, spondee, spondee, trochee**",0.200000
1,exercises-in-latin-prosody,1823,prosody_app.I.1.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so I need to figure out the f...","Trochee,Cretic,Dactyl,Trochee,Bacchius",0.600000
2,exercises-in-latin-prosody,1823,prosody_app.I.2.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's see. The user provided ...","iamb,iamb,iamb,spondee,spondee,iamb",0.833333
3,exercises-in-latin-prosody,1823,prosody_app.I.2.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's tackle this question. T...","Spondee,Trochee,Spondee,Spondee",0.250000
4,exercises-in-latin-prosody,1823,prosody_app.I.3.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's tackle this question. The...","pyrrhic, cretic, spondee",0.000000
5,exercises-in-latin-prosody,1823,prosody_app.I.3.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user is asking me to i...","spondee,trochee,iamb,iamb",0.500000
6,exercises-in-latin-prosody,1823,prosody_app.I.3.3,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's see. The user wants me to...","Trochee,Choriamb,Iamb",0.666667
7,exercises-in-latin-prosody,1823,prosody_app.I.3.4,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user provided a line o...","trochee, pyrrhic, trochee, pyrrhic",0.000000
8,exercises-in-latin-prosody,1823,prosody_app.I.4.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Choriamb, Amphibrac]","<think>\n\nOkay, let's see. The user provided ...","spondee, spondee, iamb",0.000000
9,exercises-in-latin-prosody,1823,prosody_app.I.4.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Second Epitrit, Choriamb, Choriamb, Amphibrac]","<think>\n\nOkay, so I need to figure out the m...","molossus, spondee, iamb, dactyl, dactyl, trochee",0.000000


In [170]:
# create combined df, with new columns called "source file"
for i, file in enumerate(file_to_data.keys()):
    question_df = question_dfs[i]
    if with_instr:
        file = file.replace('.json', '_with_instr.json')
    question_df['source_file'] = [file] * len(question_df)
    question_dfs[i] = question_df

combined_df = pd.concat(question_dfs)
combined_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,raw_resp,parsed_resp,score,source_file
0,exercises-in-latin-prosody,1823,prosody_app.I.1.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so the user is asking about t...","iamb, spondee, iamb, spondee, spondee, trochee**",0.200000,prosody_feet_questions_english.json
1,exercises-in-latin-prosody,1823,prosody_app.I.1.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Trochee, Spondee, Dactyl, Trochee, Trochee]","<think>\n\nOkay, so I need to figure out the f...","Trochee,Cretic,Dactyl,Trochee,Bacchius",0.600000,prosody_feet_questions_english.json
2,exercises-in-latin-prosody,1823,prosody_app.I.2.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's see. The user provided ...","iamb,iamb,iamb,spondee,spondee,iamb",0.833333,prosody_feet_questions_english.json
3,exercises-in-latin-prosody,1823,prosody_app.I.2.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[iambus, iambus, spondee, iambus]","<think>\n\nOkay, let's tackle this question. T...","Spondee,Trochee,Spondee,Spondee",0.250000,prosody_feet_questions_english.json
4,exercises-in-latin-prosody,1823,prosody_app.I.3.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's tackle this question. The...","pyrrhic, cretic, spondee",0.000000,prosody_feet_questions_english.json
5,exercises-in-latin-prosody,1823,prosody_app.I.3.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user is asking me to i...","spondee,trochee,iamb,iamb",0.500000,prosody_feet_questions_english.json
6,exercises-in-latin-prosody,1823,prosody_app.I.3.3,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, iambus]","<think>\nOkay, let's see. The user wants me to...","Trochee,Choriamb,Iamb",0.666667,prosody_feet_questions_english.json
7,exercises-in-latin-prosody,1823,prosody_app.I.3.4,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[spondee, choriambus, choriambus, iambus]","<think>\n\nOkay, so the user provided a line o...","trochee, pyrrhic, trochee, pyrrhic",0.000000,prosody_feet_questions_english.json
8,exercises-in-latin-prosody,1823,prosody_app.I.4.1,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Choriamb, Amphibrac]","<think>\n\nOkay, let's see. The user provided ...","spondee, spondee, iamb",0.000000,prosody_feet_questions_english.json
9,exercises-in-latin-prosody,1823,prosody_app.I.4.2,short_answer,scansion,unknown,english,english,What is the name of each foot in the following...,[],"[Second Epitrit, Choriamb, Choriamb, Amphibrac]","<think>\n\nOkay, so I need to figure out the m...","molossus, spondee, iamb, dactyl, dactyl, trochee",0.000000,prosody_feet_questions_english.json


In [171]:
# save combined df
save_dir = f'../data/model_responses_parsed/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# drop the columns question, multiple_choice_options, n_required, correctness_logic
save_df = combined_df.drop(columns=['question', 'multiple_choice_options', ])
save_dict = save_df.to_dict(orient='records')


save_name = 'prosody_feet.json'
if with_instr: save_name = save_name.replace('.json', '_with_instr.json')
with open(os.path.join(save_dir, save_name), 'w') as f:
    json.dump(save_dict, f, indent=4)

In [172]:
acc_lang_combos = accuracy_by(combined_df,
                             group_cols=["question_language", "answer_language"],
                             #score_cols=model_name+'_score'
                             )
acc_lang_combos

,question_language,answer_language,accuracy,n_items
0,english,english,0.256667,20
1,latin,english,0.280833,20
